# NER mit gbert-base auf dem GPTNERMED-Datensatz

In diesem Notebook trainieren wir das deutsche Modell [deepset/gbert-base](https://huggingface.co/deepset/gbert-base) auf dem [GPTNERMED-Datensatz](https://github.com/frankkramer-lab/GPTNERMED).

GPTNERMED ist ein synthetischer deutscher Datensatz für **Named Entity Recognition (NER)** im medizinischen Bereich mit drei Entitäten: `Medikation`, `Dosis` und `Diagnose`. Anders als bei der Sentiment-Klassifikation aus der Übung bekommt hier nicht der ganze Satz ein Label, sondern **jedes Token** (Token-Klassifikation, BIO-Schema).

**Wichtig:** In Colab oben unter *Laufzeit → Laufzeittyp ändern* eine **GPU (T4)** auswählen!

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q \
  "transformers>=4.40,<4.46" \
  "datasets>=2.16.0,<3.0.0" \
  "accelerate>=0.30" \
  "seqeval" \
  "evaluate>=0.4.0,<0.5.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 48.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.

In [3]:
import torch
import datasets
import transformers

print(torch.cuda.is_available())

True


## Erster Schritt: Daten laden

Wir laden den [GPTNERMED-Datensatz direkt vom Hugging Face Hub](https://huggingface.co/datasets/jfrei/GPTNERMED). Der Datensatz bringt die Splits `train` / `validation` / `test` (80/10/10) bereits mit – wir müssen also nicht mehr selbst splitten.

Auf dem Hub hat jeder Satz die Felder `sentence` und `ner_labels` (eine Liste aus `start`, `stop`, `ner_class`), z.B.:

```
{"sentence": "0,5mg Sildenafil bei erectilen Dysfunktion",
 "ner_labels": [{"start": 0, "stop": 5, "ner_class": "Dosis"},
                {"start": 6, "stop": 16, "ner_class": "Medikation"},
                {"start": 21, "stop": 42, "ner_class": "Diagnose"}]}
```

Wir formen das gleich in die Spalten `text` / `ent_starts` / `ent_stops` / `ent_classes` um, mit denen der Rest des Notebooks arbeitet.

**Hinweis:** `jfrei/GPTNERMED` ist ein Script-basierter Datensatz, deshalb ist `trust_remote_code=True` nötig (und `datasets<3.0`, wie oben installiert).

In [4]:
import datasets

# Lädt Train/Validation/Test direkt vom Hugging Face Hub
raw = datasets.load_dataset("jfrei/GPTNERMED", trust_remote_code=True)

# ner_class ist ein ClassLabel (also ints) -> zurück auf die Namen mappen
class_names = raw["train"].features["ner_labels"].feature["ner_class"].names  # ["Medikation", "Dosis", "Diagnose"]

def to_offset_format(example):
    labels = example["ner_labels"]  # dict mit Listen: start / stop / ner_class
    return {
        "text": example["sentence"],
        "ent_starts": labels["start"],
        "ent_stops": labels["stop"],
        "ent_classes": [class_names[c] for c in labels["ner_class"]],
    }

dataset = raw.map(to_offset_format, remove_columns=["sentence", "ner_labels"])

print(f"Loaded {len(dataset['train'])} train, {len(dataset['validation'])} dev and {len(dataset['test'])} test")
print(dataset["train"][2])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/7876 [00:00<?, ? examples/s]

Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Map:   0%|          | 0/985 [00:00<?, ? examples/s]

Loaded 7876 train, 984 dev and 985 test
{'text': '1.000 mg Phenprobenon über 24h, 200 mg Phenprobenon über 4 h.', 'ent_starts': [0, 9, 32, 39], 'ent_stops': [5, 21, 35, 51], 'ent_classes': ['Dosis', 'Medikation', 'Dosis', 'Medikation']}


## Zweiter Schritt: Daten vorverarbeiten

Wir definieren die Labels im **BIO-Schema**: `B-` markiert den Anfang einer Entität, `I-` die Fortsetzung, `O` bedeutet "keine Entität".

Beim Tokenisieren nutzen wir `return_offsets_mapping=True`. Damit wissen wir für jedes Token, welche Zeichen im Originaltext es abdeckt, und können so jedem Token das richtige Label aus den Zeichen-Positionen zuordnen. Spezial-Tokens wie `[CLS]` und `[SEP]` bekommen das Label `-100`, damit sie beim Training ignoriert werden.

In [5]:
from transformers import AutoTokenizer

label_list = ["O", "B-Medikation", "I-Medikation", "B-Dosis", "I-Dosis", "B-Diagnose", "I-Diagnose"]
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

tokenizer = AutoTokenizer.from_pretrained("deepset/gbert-base")

tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/362 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [6]:
def preprocess_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=128, return_offsets_mapping=True)

    all_labels = []
    for i, offsets in enumerate(tokenized["offset_mapping"]):
        labels = []
        for start, end in offsets:
            if start == end:
                labels.append(-100)
                continue
            token_label = "O"
            for j in range(len(examples["ent_starts"][i])):
                if start >= examples["ent_starts"][i][j] and end <= examples["ent_stops"][i][j]:
                    if start == examples["ent_starts"][i][j]:
                        token_label = "B-" + examples["ent_classes"][i][j]
                    else:
                        token_label = "I-" + examples["ent_classes"][i][j]
            labels.append(label2id[token_label])
        all_labels.append(labels)

    tokenized["labels"] = all_labels
    tokenized.pop("offset_mapping")
    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=["text", "ent_starts", "ent_stops", "ent_classes"])
tokenized_dataset = tokenized_dataset.shuffle(seed=42)

Map:   0%|          | 0/7876 [00:00<?, ? examples/s]

Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Map:   0%|          | 0/985 [00:00<?, ? examples/s]

In [7]:
example = tokenized_dataset["train"][0]
tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
for token, label in zip(tokens, example["labels"]):
    print(token, "->", id2label[label] if label != -100 else "-100")

[CLS] -> -100
D -> O
: -> O
Die -> O
Wund -> O
##e -> O
wird -> O
mit -> O
1 -> B-Dosis
ml -> I-Dosis
Bu -> B-Medikation
##pi -> I-Medikation
##va -> I-Medikation
##ca -> I-Medikation
##in -> I-Medikation
versorgt -> O
. -> O
[SEP] -> -100


In [8]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [9]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []
    for pred, lab in zip(predictions, labels):
        sentence_preds = []
        sentence_labels = []
        for p, l in zip(pred, lab):
            if l != -100:
                sentence_preds.append(label_list[p])
                sentence_labels.append(label_list[l])
        true_predictions.append(sentence_preds)
        true_labels.append(sentence_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    metrics = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
    # F1 pro Entitätstyp einzeln ergänzen (seqeval liefert diese unter den Klassennamen)
    for entity in ["Medikation", "Dosis", "Diagnose"]:
        if entity in results:
            metrics[f"{entity}_f1"] = results[entity]["f1"]
    return metrics

## Dritter Schritt: Training (über drei Seeds)

Jetzt laden wir `gbert-base` mit einem Klassifikations-Kopf für Token-Klassifikation (7 Labels) und trainieren mit dem `Trainer`.

Um zu sehen, wie **stabil** die Ergebnisse sind, trainieren wir das Modell **dreimal mit unterschiedlichen Zufalls-Seeds** (`42`, `123`, `2024`). Der Seed steuert u.a. die Initialisierung des Klassifikations-Kopfes und die Reihenfolge der Trainingsdaten. Am Ende können wir so Mittelwert und Standardabweichung der Test-Metriken berechnen, statt uns auf einen einzelnen (womöglich glücklichen oder unglücklichen) Lauf zu verlassen.

Auf einer T4-GPU dauert **ein** Durchlauf ca. 10-15 Minuten, alle drei Seeds zusammen also ca. **30-45 Minuten**.

In [11]:
import wandb
wandb.init(mode="disabled")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


In [13]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, set_seed

seeds = [42, 52, 62]
seed_results = []  # sammelt die Test-Metriken pro Seed

for seed in seeds:
    print(f"\n===== Training mit Seed {seed} =====")
    set_seed(seed)  # macht u.a. die Gewichts-Init des Klassifikations-Kopfes reproduzierbar

    model = AutoModelForTokenClassification.from_pretrained(
        "deepset/gbert-base", num_labels=7, id2label=id2label, label2id=label2id
    )

    training_args = TrainingArguments(
    output_dir=f"gbert_gptnermed_seed{seed}",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # Test-Metriken für diesen Seed sichern
    test_metrics = trainer.predict(tokenized_dataset["test"]).metrics
    test_metrics["seed"] = seed
    seed_results.append(test_metrics)
    print(f"Seed {seed}: Test-F1 = {test_metrics['test_f1']:.4f}")

    """
    # Bestes Modell dieses Seeds abspeichern (durch load_best_model_at_end=True ist
    # trainer.model bereits das beste Modell aus dem Training). Landet im Colab-Dateisystem.
    save_dir = f"/content/drive/MyDrive/gbert-base-no-hp-seed{seed}"
    trainer.save_model(save_dir)          # speichert Modell-Gewichte + Config
    tokenizer.save_pretrained(save_dir)   # speichert den Tokenizer dazu
    print(f"Seed {seed}: bestes Modell gespeichert unter '{save_dir}'")
    """

print("\nAlle Seeds trainiert!")

# Letztes trainiertes Modell für Test-Auswertung und Inferenz behalten
model = trainer.model.module if hasattr(trainer.model, 'module') else trainer.model


===== Training mit Seed 42 =====


Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.287700,0.191489,0.849572,0.884926,0.866889,0.940498
2,0.137600,0.189246,0.859734,0.905732,0.882134,0.942131
3,0.071400,0.215683,0.881081,0.899788,0.890336,0.945550


Seed 42: Test-F1 = 0.8974

===== Training mit Seed 52 =====


Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.280900,0.191157,0.852127,0.876008,0.863903,0.938967
2,0.135800,0.192795,0.849238,0.899363,0.873582,0.939273
3,0.070500,0.205658,0.889823,0.898514,0.894147,0.946060


Some weights of BertForTokenClassification were not initialized from the model checkpoint at deepset/gbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Seed 52: Test-F1 = 0.8951

===== Training mit Seed 62 =====


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.285400,0.194325,0.831335,0.883227,0.856496,0.936518
2,0.140000,0.186901,0.858643,0.902760,0.880149,0.940498
3,0.071000,0.205239,0.879153,0.898938,0.888936,0.944785


Seed 62: Test-F1 = 0.8987

Alle Seeds trainiert!


In [15]:
import numpy as np

print("Test-Ergebnisse pro Seed:")
for r in seed_results:
    print(f"  Seed {r['seed']}: F1={r['test_f1']:.4f}  Precision={r['test_precision']:.4f}  Recall={r['test_recall']:.4f}")

f1_scores = [r["test_f1"] for r in seed_results]
prec_scores = [r["test_precision"] for r in seed_results]
rec_scores = [r["test_recall"] for r in seed_results]

print("\nÜber alle Seeds gemittelt (Mittelwert ± Standardabweichung):")
print(f"  F1:        {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Precision: {np.mean(prec_scores):.4f} ± {np.std(prec_scores):.4f}")
print(f"  Recall:    {np.mean(rec_scores):.4f} ± {np.std(rec_scores):.4f}")

Test-Ergebnisse pro Seed:
  Seed 42: F1=0.8974  Precision=0.8877  Recall=0.9072
  Seed 52: F1=0.8951  Precision=0.8856  Recall=0.9047
  Seed 62: F1=0.8987  Precision=0.8940  Recall=0.9034

Über alle Seeds gemittelt (Mittelwert ± Standardabweichung):
  F1:        0.8970 ± 0.0015
  Precision: 0.8891 ± 0.0035
  Recall:    0.9051 ± 0.0016
